# Baseline: медиана и TF-IDF с логистической регрессией

Цель — предсказать оценку отзыва от 1 до 5. Сравним постоянный прогноз с текстовой моделью, проверим признаки из EDA и выберем настройку по MAE на пяти разбиениях. `test.csv` Kaggle не содержит известных оценок и в этих расчётах не участвует.

Все словари и параметры масштабирования обучаются только на обучающей части соответствующего разбиения.

In [1]:
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def project_root() -> Path:
    current = Path.cwd().resolve()

    for parent in (current, *current.parents):
        if (parent / "data" / "raw" / "train.csv").exists():
            return parent

    raise FileNotFoundError("Поместите train.csv в data/raw/ проекта")


root = project_root()
sys.path.insert(0, str(root))

from company_reviews.baseline import (
    BaselineModel,
    combine_features,
    feature_groups,
    normalize_reviews_for_split,
    median_from_probabilities,
    numeric_feature_matrix,
)

train = pd.read_csv(root / "data" / "raw" / "train.csv")
if list(train.columns) != ["Id", "Review", "Rating"]:
    raise ValueError("Ожидаются столбцы Id, Review, Rating")
if train["Review"].isna().any() or not train["Rating"].isin(range(1, 6)).all():
    raise ValueError("В Review есть пропуски или Rating выходит за диапазон 1–5")

print(f"Строк: {len(train):,}; пропусков в Review: {train['Review'].isna().sum()}")

Строк: 60,000; пропусков в Review: 0


## 1. Какие данные загружены

Покажем по одному отзыву каждой оценки. В таблице текст сокращён только для просмотра; модель получит полный `Review`. Столбец `Id` нужен для сопоставления строк, но не входит в признаки.

In [2]:
preview = (
    train.groupby("Rating", sort=True)
    .head(1)[["Id", "Review", "Rating"]]
    .sort_values("Rating")
    .copy()
)
preview["Review"] = preview["Review"].str.slice(0, 160)
display(preview.reset_index(drop=True))

,Id,Review,Rating
0,2,I couldn't make an official review on a produc...,1
1,17,"Great product, seemed to take an age to be del...",2
2,1,After 6 month still can't access my account,3
3,15,I got a couple of 30 day sim cards for my sons...,4
4,0,Very good value and a great tv very happy and ...,5


## 2. Повторы и разделение данных

Сначала удаляем строки с **точно таким же `Review`**: иначе повторный отзыв получил бы дополнительный вес при обучении. У этих повторов оценки совпадают.

Чтобы не разделить отзывы, совпадающие после нормализации, между обучением и проверкой, создаём служебную копию каждого оставшегося текста: приводим буквы к одному регистру и схлопываем пробелы. Если две такие копии равны, присваиваем отзывам **один номер группы**. Это не группировка по теме, смыслу или оценке. Номер нужен только при разделении данных; модель получает исходный текст `Review`.

Ниже показаны число групп и единственная пара отзывов, которая совпала после такой нормализации.

In [3]:
duplicate_labels = (
    train.loc[train.duplicated("Review", keep=False)]
    .groupby("Review")["Rating"]
    .nunique()
)
assert (duplicate_labels <= 1).all()

train_clean = train.drop_duplicates(subset="Review", keep="first").reset_index(drop=True)
normalized_reviews = normalize_reviews_for_split(train_clean["Review"])
group_numbers = pd.Series(
    pd.factorize(normalized_reviews, sort=True)[0] + 1,
    index=train_clean.index,
    name="номер группы",
)
group_sizes = group_numbers.value_counts()

print(f"Удалено строк с повторным Review: {len(train) - len(train_clean)}")
print(f"Осталось отзывов: {len(train_clean):,}")
print(f"Групп: {len(group_sizes):,}")
print(f"Групп из одного отзыва: {(group_sizes == 1).sum():,}")
print(f"Групп из двух и более отзывов: {(group_sizes > 1).sum():,}")

paired_rows = group_numbers[group_numbers.duplicated(keep=False)].index
example = train_clean.loc[paired_rows, ["Review", "Rating"]].copy()
example["после нормализации"] = normalized_reviews.loc[paired_rows]
example["номер группы"] = group_numbers.loc[paired_rows]
display(example.reset_index(drop=True))

Удалено строк с повторным Review: 24
Осталось отзывов: 59,976
Групп: 59,975
Групп из одного отзыва: 59,974
Групп из двух и более отзывов: 1


,Review,Rating,после нормализации,номер группы
0,"Great product, delivered on time.",5,"great product, delivered on time.",20124
1,"great product, delivered on time.",5,"great product, delivered on time.",20124


Обе строки в таблице получили один номер группы. Сами номера произвольны: важно только, совпадают они или нет. Здесь групп из нескольких отзывов всего одна. Такое разделение не даст двум вариантам этого отзыва оказаться по разные стороны проверки.

`StratifiedGroupKFold` по возможности сохраняет доли оценок 1–5 и держит отзывы одной группы вместе. Фолд — одна из пяти проверочных частей. Для коротких опытов берём первый фолд и затем ещё раз делим его обучающую часть. В итоговом сравнении каждый из пяти фолдов по очереди станет проверочным.

In [4]:
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2026)
train_idx, valid_idx = next(
    splitter.split(train_clean["Review"], train_clean["Rating"], group_numbers)
)

train_part = train_clean.iloc[train_idx]
valid_part = train_clean.iloc[valid_idx]
assert set(group_numbers.iloc[train_idx]).isdisjoint(set(group_numbers.iloc[valid_idx]))

print(f"Обучение: {len(train_part):,}; проверка: {len(valid_part):,}")
class_share = pd.DataFrame({
    "обучение, %": 100 * train_part["Rating"].value_counts(normalize=True).sort_index(),
    "проверка, %": 100 * valid_part["Rating"].value_counts(normalize=True).sort_index(),
})
display(class_share.round(2).rename_axis("Rating"))

Обучение: 47,980; проверка: 11,996


,"обучение, %","проверка, %"
Rating,,
1,31.11,31.11
2,2.72,2.72
3,2.80,2.80
4,5.58,5.59
5,57.79,57.79


## 3. Постоянный прогноз: медиана

Для постоянного прогноза $c$ сумма абсолютных ошибок равна

\[
L(c)=\sum_{i=1}^{n}|y_i-c|.
\]

Если оценки целые, то

\[
L(c+1)-L(c)=\#\{y_i\leq c\}-\#\{y_i\geq c+1\}.
\]

Пока справа больше оценок, сдвиг $c$ вправо уменьшает ошибку. Когда слева их становится больше, ошибка растёт. Граница между этими случаями — медиана.

In [5]:
y_train = train_part["Rating"].to_numpy()
y_valid = valid_part["Rating"].to_numpy()

dummy = DummyRegressor(strategy="median")
dummy.fit(np.zeros((len(train_part), 1)), y_train)
dummy_pred = dummy.predict(np.zeros((len(valid_part), 1)))
dummy_mae = mean_absolute_error(y_valid, dummy_pred)

print(f"Постоянный прогноз: {dummy_pred[0]:.0f}")
print(f"MAE на проверке: {dummy_mae:.5f}")

Постоянный прогноз: 5
MAE на проверке: 1.43781


## 4. Первая текстовая модель

TF-IDF превращает слова и пары слов в числовые признаки. `min_df=3` — начальный порог частоты; ниже сравним его с другими значениями. Ограничение `max_features=150_000` задаёт верхнюю границу словаря, а `sublinear_tf=True` ослабляет влияние многократного повтора слова. Начальное `C=4` проверим позднее.

Логистическая регрессия выдаёт вероятности оценок 1–5. Ответом берём медиану этого распределения: первую оценку, на которой накопленная вероятность достигает 0,5. Она минимизирует ожидаемую абсолютную ошибку для полученных вероятностей:

\[
\hat y = \min\left\{r\in\{1,\ldots,5\}: \sum_{k\le r}p_k\ge 0{,}5\right\}.
\]

In [6]:
word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_features=150_000,
    sublinear_tf=True,
    dtype=np.float32,
)
word_classifier = LogisticRegression(C=4, max_iter=250, solver="lbfgs")
word_model = Pipeline([
    ("tfidf", word_vectorizer),
    ("classifier", word_classifier),
])
word_model.fit(train_part["Review"], y_train)

probabilities = word_model.predict_proba(valid_part["Review"])
classes = word_model.named_steps["classifier"].classes_
word_pred = median_from_probabilities(probabilities, classes)
word_mae = mean_absolute_error(y_valid, word_pred)

display(pd.DataFrame({
    "модель": ["Постоянная медиана", "Слова TF-IDF + логистическая регрессия"],
    "MAE": [dummy_mae, word_mae],
}).round(5))

,модель,MAE
0,Постоянная медиана,1.43781
1,Слова TF-IDF + логистическая регрессия,0.19165


На первом разбиении текстовая модель лучше постоянного прогноза. Отдельно проверим оценки 2–4: их мало, и общая MAE может скрывать ошибки на этих классах.

In [7]:
rating_summary = pd.DataFrame({
    "отзывов": [(y_valid == rating).sum() for rating in classes],
    "MAE": [
        mean_absolute_error(y_valid[y_valid == rating], word_pred[y_valid == rating])
        for rating in classes
    ],
}, index=pd.Index(classes, name="Rating"))
display(rating_summary.round(4))

,отзывов,MAE
Rating,,
1,3732,0.1294
2,326,1.0000
3,336,1.2976
4,670,0.8940
5,6932,0.0656


## 5. Короткие опыты с признаками

Для них делим обучающую часть первого фолда ещё раз. Во всех таблицах этого раздела обучающие и проверочные отзывы одинаковы; меняется только указанная настройка или набор признаков. Следующий раздел перепроверит кандидатов на всех пяти фолдах.

In [8]:
inner_part = train_part.reset_index(drop=True)
inner_group_numbers = group_numbers.iloc[train_idx].reset_index(drop=True)
inner_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2027)
fit_idx, check_idx = next(
    inner_splitter.split(inner_part["Review"], inner_part["Rating"], inner_group_numbers)
)

fit_part = inner_part.iloc[fit_idx]
check_part = inner_part.iloc[check_idx]
assert set(inner_group_numbers.iloc[fit_idx]).isdisjoint(set(inner_group_numbers.iloc[check_idx]))
print(f"Внутреннее обучение: {len(fit_part):,}; проверка: {len(check_part):,}")

Внутреннее обучение: 38,384; проверка: 9,596


### 5.1. Стоп-слова

Английский список содержит отрицания, способные менять смысл отзыва. Сравним сохранение всех слов, удаление списка и более мягкие варианты. `max_df=0.3` удаляет термины, встречающиеся более чем в 30% обучающих отзывов.

In [9]:
negations = {"no", "not", "never", "nor"}
context_words = negations | {"but", "however", "very", "without"}
stopword_options = [
    ("Все слова", None, 1.0),
    ("Удалены английские стоп-слова", "english", 1.0),
    ("Стоп-слова удалены, отрицания сохранены", sorted(ENGLISH_STOP_WORDS - negations), 1.0),
    ("Сохранены отрицания и связки", sorted(ENGLISH_STOP_WORDS - context_words), 1.0),
    ("Удалены термины из >30% отзывов", None, 0.3),
]

stopword_rows = []
for name, stop_words, max_df in stopword_options:
    candidate = clone(word_model)
    candidate.set_params(tfidf__stop_words=stop_words, tfidf__max_df=max_df)
    candidate.fit(fit_part["Review"], fit_part["Rating"])

    probabilities = candidate.predict_proba(check_part["Review"])
    labels = candidate.named_steps["classifier"].classes_
    prediction = median_from_probabilities(probabilities, labels)
    stopword_rows.append({
        "вариант": name,
        "MAE": mean_absolute_error(check_part["Rating"], prediction),
    })

display(pd.DataFrame(stopword_rows).round(5))

,вариант,MAE
0,Все слова,0.19675
1,Удалены английские стоп-слова,0.22614
2,"Стоп-слова удалены, отрицания сохранены",0.21728
3,Сохранены отрицания и связки,0.20873
4,Удалены термины из >30% отзывов,0.19644


Список стоп-слов в дальнейшей модели не применяем: его удаление ухудшило MAE. Небольшое преимущество `max_df=0.3` на одном разбиении не считаем достаточным основанием менять базовый вариант.

### 5.2. Длина и пунктуация из EDA

EDA показал различия по длине отзыва и знакам `!`/`?`. Проверим четыре признака: логарифмы длины в символах и словах, наличие `!` и наличие `?`. Их масштаб рассчитывается только по обучающим отзывам. Вес 0,1 пока фиксирован; ниже проверим соседние значения.

In [10]:
def score_numeric_variant(
    fit_text,
    check_text,
    fit_numeric: np.ndarray,
    check_numeric: np.ndarray,
    fit_labels: pd.Series,
    check_labels: pd.Series,
    *,
    c: float = 4,
    numeric_weight: float = 0.1,
) -> float:
    scaler = StandardScaler()
    fit_scaled = scaler.fit_transform(fit_numeric)
    check_scaled = scaler.transform(check_numeric)

    x_fit = combine_features(fit_text, None, fit_scaled, 0, numeric_weight)
    x_check = combine_features(check_text, None, check_scaled, 0, numeric_weight)

    classifier = LogisticRegression(C=c, max_iter=250, solver="lbfgs")
    classifier.fit(x_fit, fit_labels)
    prediction = median_from_probabilities(
        classifier.predict_proba(x_check), classifier.classes_
    )
    return mean_absolute_error(check_labels, prediction)

In [11]:
inner_word = clone(word_model)
inner_word.fit(fit_part["Review"], fit_part["Rating"])

fit_text = inner_word.named_steps["tfidf"].transform(fit_part["Review"])
check_text = inner_word.named_steps["tfidf"].transform(check_part["Review"])
inner_pred = median_from_probabilities(
    inner_word.predict_proba(check_part["Review"]),
    inner_word.named_steps["classifier"].classes_,
)
inner_word_mae = mean_absolute_error(check_part["Rating"], inner_pred)

fit_eda = numeric_feature_matrix(fit_part["Review"])[:, :4]
check_eda = numeric_feature_matrix(check_part["Review"])[:, :4]
eda_mae = score_numeric_variant(
    fit_text, check_text, fit_eda, check_eda,
    fit_part["Rating"], check_part["Rating"],
)

display(pd.DataFrame({
    "признаки": ["Только слова", "Слова + 4 признака EDA"],
    "MAE": [inner_word_mae, eda_mae],
}).round(5))

,признаки,MAE
0,Только слова,0.19675
1,Слова + 4 признака EDA,0.19633


### 5.3. Другие возможные сигналы

Проверим слова прописными буквами, количество знаков `!`/`?` и явно написанную оценку вроде `one star` или `5/5`. К четырём признакам EDA добавляются 1 + 2 + 6 признаков соответственно: всего 13. Их извлечение находится в `company_reviews/baseline.py`, чтобы тот же код можно было использовать при прогнозе нового отзыва. Стемминг и лемматизацию сейчас не добавляем; ниже отдельно проверим символьные фрагменты слов.

In [12]:
feature_examples = pd.Series(["WOW!! one star?", "great company 5/5"])
example_features = numeric_feature_matrix(feature_examples)
display(pd.DataFrame({
    "отзыв": feature_examples,
    "прописных слов": np.expm1(example_features[:, 4]).astype(int),
    "знаков !": np.expm1(example_features[:, 5]).astype(int),
    "знаков ?": np.expm1(example_features[:, 6]).astype(int),
    "найдена оценка": np.where(
        example_features[:, 7:13].any(axis=1),
        example_features[:, 7:13].argmax(axis=1),
        "нет",
    ),
}))

mentioned_flags = numeric_feature_matrix(train_clean["Review"])[:, 7:13]
print(f"Отзывов с распознанной оценкой: {int(mentioned_flags.any(axis=1).sum()):,}")

,отзыв,прописных слов,знаков !,знаков ?,найдена оценка
0,WOW!! one star?,1,2,1,1
1,great company 5/5,0,0,0,5


Отзывов с распознанной оценкой: 1,792


In [13]:
feature_options = [
    ("Только 4 признака EDA", ["EDA"]),
    ("+ прописные слова", ["EDA", "прописные слова"]),
    ("+ количество ! и ?", ["EDA", "число ! и ?"]),
    ("+ упоминание оценки", ["EDA", "упоминание оценки"]),
    ("+ все три группы", list(feature_groups(fit_part["Review"]))),
]

fit_groups = feature_groups(fit_part["Review"])
check_groups = feature_groups(check_part["Review"])
feature_rows = []

for name, selected in feature_options:
    fit_numeric = np.column_stack([fit_groups[key] for key in selected])
    check_numeric = np.column_stack([check_groups[key] for key in selected])
    score = score_numeric_variant(
        fit_text, check_text, fit_numeric, check_numeric,
        fit_part["Rating"], check_part["Rating"],
    )
    feature_rows.append({"набор признаков": name, "MAE": score})

display(pd.DataFrame(feature_rows).round(5))

,набор признаков,MAE
0,Только 4 признака EDA,0.19633
1,+ прописные слова,0.19633
2,+ количество ! и ?,0.19696
3,+ упоминание оценки,0.19539
4,+ все три группы,0.19446


Полный набор из 13 числовых признаков оказался лучшим на внутреннем разбиении. Это предварительный выбор: в разделе 6 сравним его с меньшими наборами на пяти фолдах при одинаковых настройках модели.

### 5.4. Небольшой сеточный поиск

На одном внутреннем разбиении вручную проверим все 12 сочетаний `C` из {1, 2, 4, 8} и веса числовых признаков из {0,05; 0,1; 0,2}. Это **сетка по двум параметрам**, а не перебор всех настроек TF-IDF и модели. Меньшее `C` сильнее ограничивает коэффициенты логистической регрессии.

In [14]:
fit_numeric = numeric_feature_matrix(fit_part["Review"])
check_numeric = numeric_feature_matrix(check_part["Review"])
parameter_rows = []

for c in (1, 2, 4, 8):
    for numeric_weight in (0.05, 0.1, 0.2):
        score = score_numeric_variant(
            fit_text, check_text, fit_numeric, check_numeric,
            fit_part["Rating"], check_part["Rating"],
            c=c, numeric_weight=numeric_weight,
        )
        parameter_rows.append({"C": c, "вес числовых": numeric_weight, "MAE": score})

parameter_results = pd.DataFrame(parameter_rows)
display(parameter_results.pivot(index="C", columns="вес числовых", values="MAE").round(5))

вес числовых,0.05,0.10,0.20
C,,,
1,0.21415,0.21207,0.21321
2,0.20113,0.20238,0.20300
4,0.19352,0.19446,0.19498
8,0.19206,0.19185,0.19195


In [15]:
best_inner = parameter_results.sort_values("MAE").iloc[0]
inner_c = float(best_inner["C"])
numeric_weight = float(best_inner["вес числовых"])

print(f"Для следующей проверки: C={inner_c:g}; вес числовых={numeric_weight:g}")

Для следующей проверки: C=8; вес числовых=0.1


## 6. Проверка на пяти фолдах

Сначала сравним полные и сокращённые наборы признаков при одинаковых `min_df=5` и `C=4`. Рядом проверим пороги `min_df` 2, 3, 5 и 10, символьные n-граммы длиной 3–5 и мягкие веса редких классов. Символьные фрагменты могут уловить формы слов и опечатки без стемминга или лемматизации. Для вариантов, найденных на внутреннем разбиении, используем `C=8`; прежний `C=4` остаётся прямым контрольным вариантом.

Далее сравним 13 заранее выбранных конфигураций на пяти фолдах. Это уже не полная сетка всех параметров: каждая строка задаёт конкретное сочетание, а отдельные поля показывают различия. Во всех фолдах словари TF-IDF и масштабирование рассчитываются заново на обучающей части. «Разброс» в таблицах — стандартное отклонение MAE между пятью фолдами.

Для весов классов используем долю оценки $f_r$ только в обучающей части фолда:

$$w_r = \frac{f_r^{-p}}{\sum_k f_k^{1-p}}.$$

При $p=0$ все веса равны 1. При $p=0{,}1$ или $0{,}25$ редкие оценки получают больший вес, но мягче, чем при полном обратном взвешивании; знаменатель оставляет средний вес обучающего примера равным 1. В таблицах $p$ назван «степень веса классов».

In [ ]:
def present_configs(table: pd.DataFrame) -> pd.DataFrame:
    """Понятные названия в таблицах; внутренние имена остаются в коде."""
    readable = table.copy()
    readable["numeric"] = readable["numeric"].replace({
        "нет": "нет",
        "4 EDA": "4 из EDA",
        "13": "все 13",
    })
    readable = readable.rename(columns={
        "id": "вариант",
        "text": "текстовые признаки",
        "numeric": "числовые признаки",
        "c": "C",
        "char": "вес символов",
        "power": "степень веса классов",
    })

    if readable.index.name == "id":
        readable.index.name = "вариант"

    return readable

In [16]:
configurations = [
    # Прямое сравнение наборов признаков: одинаковые min_df и C.
    dict(id="word", text="слова", numeric="нет", min_df=5, c=4, char=0, power=0),
    dict(id="word_eda", text="слова", numeric="4 EDA", min_df=5, c=4, char=0, power=0),
    dict(id="word_all", text="слова", numeric="13", min_df=5, c=4, char=0, power=0),
    dict(
        id="word_char", text="слова + символы", numeric="нет",
        min_df=5, c=4, char=0.5, power=0,
    ),
    dict(
        id="full", text="слова + символы", numeric="13",
        min_df=5, c=4, char=0.5, power=0,
    ),

    # Проверка настройки при указанных для каждого варианта наборах признаков.
    dict(id="df2", text="слова", numeric="13", min_df=2, c=8, char=0, power=0),
    dict(id="df3", text="слова", numeric="13", min_df=3, c=8, char=0, power=0),
    dict(id="df5", text="слова", numeric="13", min_df=5, c=8, char=0, power=0),
    dict(id="df10", text="слова", numeric="13", min_df=10, c=8, char=0, power=0),
    dict(
        id="char_df3", text="слова + символы", numeric="13",
        min_df=3, c=8, char=0.5, power=0,
    ),
    dict(
        id="char_df5", text="слова + символы", numeric="13",
        min_df=5, c=8, char=0.5, power=0,
    ),
    dict(
        id="weighted_01", text="слова + символы", numeric="13",
        min_df=5, c=8, char=0.5, power=0.1,
    ),
    dict(
        id="weighted_025", text="слова + символы", numeric="13",
        min_df=5, c=8, char=0.5, power=0.25,
    ),
]

config_by_id = {config["id"]: config for config in configurations}
assert len(config_by_id) == len(configurations)
display(present_configs(pd.DataFrame(configurations)))

,вариант,текстовые признаки,числовые признаки,min_df,C,вес символов,степень веса классов
0,word,слова,нет,5,4,0.0,0.00
1,word_eda,слова,4 из EDA,5,4,0.0,0.00
2,word_all,слова,все 13,5,4,0.0,0.00
3,word_char,слова + символы,нет,5,4,0.5,0.00
4,full,слова + символы,все 13,5,4,0.5,0.00
5,df2,слова,все 13,2,8,0.0,0.00
6,df3,слова,все 13,3,8,0.0,0.00
7,df5,слова,все 13,5,8,0.0,0.00
8,df10,слова,все 13,10,8,0.0,0.00
9,char_df3,слова + символы,все 13,3,8,0.5,0.00


In [17]:
def class_weights(labels: np.ndarray, power: float) -> dict[int, float]:
    frequencies = {rating: np.mean(labels == rating) for rating in range(1, 6)}
    raw = {rating: frequency ** (-power) for rating, frequency in frequencies.items()}
    average = sum(frequencies[rating] * raw[rating] for rating in frequencies)
    return {rating: weight / average for rating, weight in raw.items()}


def numeric_views(fit_reviews: pd.Series, check_reviews: pd.Series):
    scaler = StandardScaler()
    fit_full = scaler.fit_transform(numeric_feature_matrix(fit_reviews))
    check_full = scaler.transform(numeric_feature_matrix(check_reviews))

    return {
        "нет": (None, None),
        "4 EDA": (fit_full[:, :4], check_full[:, :4]),
        "13": (fit_full, check_full),
    }

In [18]:
def evaluate_candidate(
    config: dict,
    fit_word,
    check_word,
    fit_char,
    check_char,
    numeric: dict,
    y_fit: np.ndarray,
    y_check: np.ndarray,
) -> tuple[float, np.ndarray]:
    fit_numeric, check_numeric = numeric[config["numeric"]]
    fit_matrix = combine_features(
        fit_word, fit_char, fit_numeric, config["char"], numeric_weight
    )
    check_matrix = combine_features(
        check_word, check_char, check_numeric, config["char"], numeric_weight
    )

    power = config["power"]
    classifier = LogisticRegression(
        C=config["c"],
        max_iter=250,
        solver="lbfgs",
        class_weight=class_weights(y_fit, power) if power else None,
    )
    classifier.fit(fit_matrix, y_fit)
    prediction = median_from_probabilities(
        classifier.predict_proba(check_matrix), classifier.classes_
    )

    return mean_absolute_error(y_check, prediction), prediction

Сначала задаём расчёт для одного фолда. Словарь слов обучается заново для каждого порога min_df; словарь символов и масштабирование числовых признаков — один раз на обучающей части фолда. Затем запускаем одинаковую процедуру для всех пяти фолдов.

In [19]:
configs_by_min_df = {
    threshold: [
        config for config in configurations if config["min_df"] == threshold
    ]
    for threshold in sorted({config["min_df"] for config in configurations})
}


def evaluate_fold(fold: int, fit_idx, check_idx):
    fit = train_clean.iloc[fit_idx]
    check = train_clean.iloc[check_idx]
    y_fit = fit["Rating"].to_numpy()
    y_check = check["Rating"].to_numpy()

    assert set(group_numbers.iloc[fit_idx]).isdisjoint(set(group_numbers.iloc[check_idx]))

    naive = DummyRegressor(strategy="median")
    naive.fit(np.zeros((len(fit), 1)), y_fit)
    naive_prediction = naive.predict(np.zeros((len(check), 1))).astype(np.int8)
    rows = [{
        "фолд": fold,
        "id": "median",
        "MAE": mean_absolute_error(y_check, naive_prediction),
    }]
    predictions = {}

    numeric = numeric_views(fit["Review"], check["Review"])
    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=5,
        max_features=100_000,
        sublinear_tf=True,
        dtype=np.float32,
    )
    fit_char = char_vectorizer.fit_transform(fit["Review"])
    check_char = char_vectorizer.transform(check["Review"])

    for min_df, candidates in configs_by_min_df.items():
        word_vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=min_df,
            max_features=150_000,
            sublinear_tf=True,
            dtype=np.float32,
        )
        fit_word = word_vectorizer.fit_transform(fit["Review"])
        check_word = word_vectorizer.transform(check["Review"])

        for config in candidates:
            mae, prediction = evaluate_candidate(
                config, fit_word, check_word, fit_char, check_char,
                numeric, y_fit, y_check,
            )
            predictions[config["id"]] = prediction
            rows.append({"фолд": fold, "id": config["id"], "MAE": mae})

    return rows, naive_prediction, predictions

In [20]:
cv_rows = []
oof_predictions = {
    config["id"]: np.full(len(train_clean), -1, dtype=np.int8)
    for config in configurations
}
dummy_predictions = np.full(len(train_clean), -1, dtype=np.int8)
fold_coverage = np.zeros(len(train_clean), dtype=np.int8)

for fold, (fit_idx, check_idx) in enumerate(
    splitter.split(train_clean["Review"], train_clean["Rating"], group_numbers),
    start=1,
):
    rows, naive_prediction, predictions = evaluate_fold(fold, fit_idx, check_idx)
    cv_rows.extend(rows)
    dummy_predictions[check_idx] = naive_prediction
    fold_coverage[check_idx] += 1

    for config_id, prediction in predictions.items():
        oof_predictions[config_id][check_idx] = prediction

    print(f"Фолд {fold}/5: проверено {len(check_idx):,} отзывов")

assert np.all(fold_coverage == 1)
assert all(np.all(pred != -1) for pred in oof_predictions.values())

Фолд 1/5: проверено 11,996 отзывов


Фолд 2/5: проверено 11,996 отзывов


Фолд 3/5: проверено 11,995 отзывов


Фолд 4/5: проверено 11,995 отзывов


Фолд 5/5: проверено 11,994 отзывов


In [21]:
cv_metrics = pd.DataFrame(cv_rows)
mean_scores = (
    cv_metrics.groupby("id", as_index=False)["MAE"]
    .agg(["mean", "std"])
    .rename(columns={"mean": "средняя MAE", "std": "разброс"})
)
cv_summary = (
    pd.DataFrame(configurations)
    .merge(mean_scores, on="id")
    .sort_values("средняя MAE")
)

ablation_ids = ["word", "word_eda", "word_all", "word_char", "full"]
ablation = cv_summary.set_index("id").loc[ablation_ids]
display(present_configs(ablation[["text", "numeric", "средняя MAE", "разброс"]].round(5)))

,текстовые признаки,числовые признаки,средняя MAE,разброс
вариант,,,,
word,слова,нет,0.19008,0.00469
word_eda,слова,4 из EDA,0.18946,0.00459
word_all,слова,все 13,0.18751,0.00496
word_char,слова + символы,нет,0.18869,0.00396
full,слова + символы,все 13,0.18539,0.00498


В таблице выше меняется только набор признаков: `min_df=5`, `C=4` и фолды одинаковы. Добавление четырёх признаков EDA к словам снижает среднюю MAE с 0,19008 до 0,18946; весь числовой набор — до 0,18751. Символьные фрагменты без числовых признаков дают 0,18869, вместе с ними — 0,18539. Поэтому оставляем оба дополнения к словам.

In [22]:
tuning = cv_summary[~cv_summary["id"].isin(ablation_ids)]
display(present_configs(tuning[[
    "text", "numeric", "min_df", "c", "char", "power",
    "средняя MAE", "разброс",
]].round(5).reset_index(drop=True)))

selected_id = cv_summary.iloc[0]["id"]
selected_config = config_by_id[selected_id]
print(f"Лучший вариант раздела 6: {selected_id}")
print(f"Средняя MAE: {cv_summary.iloc[0]['средняя MAE']:.5f}")
median_cv_mae = mean_scores.loc[
    mean_scores["id"] == "median", "средняя MAE"
].iloc[0]
print(f"Постоянная медиана: {median_cv_mae:.5f}")

,текстовые признаки,числовые признаки,min_df,C,вес символов,степень веса классов,средняя MAE,разброс
0,слова + символы,все 13,3,8,0.5,0.00,0.18614,0.00459
1,слова + символы,все 13,5,8,0.5,0.00,0.18676,0.00450
2,слова,все 13,2,8,0.0,0.00,0.18726,0.00527
3,слова,все 13,3,8,0.0,0.00,0.18746,0.00482
4,слова,все 13,5,8,0.0,0.00,0.18793,0.00486
5,слова + символы,все 13,5,8,0.5,0.10,0.18824,0.00422
6,слова,все 13,10,8,0.0,0.00,0.19008,0.00414
7,слова + символы,все 13,5,8,0.5,0.25,0.19101,0.00537


Лучший вариант раздела 6: full
Средняя MAE: 0.18539
Постоянная медиана: 1.43781


Посмотрим ошибки по отдельным оценкам. Каждый отзыв получил один прогноз на том фолде, где он был проверочным. Сравнение с вариантом «только слова» использует те же пять фолдов.

In [23]:
y_all = train_clean["Rating"].to_numpy()
rating_errors = pd.DataFrame({
    "отзывов": [(y_all == rating).sum() for rating in range(1, 6)],
    "постоянная медиана": [
        mean_absolute_error(y_all[y_all == rating], dummy_predictions[y_all == rating])
        for rating in range(1, 6)
    ],
    "только слова": [
        mean_absolute_error(y_all[y_all == rating], oof_predictions["word"][y_all == rating])
        for rating in range(1, 6)
    ],
    "лучший вариант раздела 6": [
        mean_absolute_error(y_all[y_all == rating], oof_predictions[selected_id][y_all == rating])
        for rating in range(1, 6)
    ],
}, index=pd.Index(range(1, 6), name="Rating"))
display(rating_errors.round(4))

fold_scores = cv_metrics.pivot(index="фолд", columns="id", values="MAE")
display(fold_scores[["word", selected_id]].round(5))

,отзывов,постоянная медиана,только слова,лучший вариант раздела 6
Rating,,,,
1,18660,4.0,0.1247,0.1268
2,1629,3.0,1.0516,1.0135
3,1679,2.0,1.2734,1.2245
4,3349,1.0,0.8546,0.8325
5,34659,0.0,0.0681,0.0651


id,word,full
фолд,,
1,0.19348,0.18881
2,0.18773,0.17898
3,0.19642,0.19158
4,0.18766,0.18258
5,0.18509,0.18501


## 7. GridSearchCV: совместная настройка C и min_df

Выше мы сравнивали признаки и несколько отдельных настроек. Теперь фиксируем полный набор: слова, символы с весом 0,5 и 13 числовых признаков с весом 0,1. Проверяем **все 9 сочетаний** `C` ∈ {2, 4, 8} и `min_df` ∈ {2, 3, 5} на тех же пяти фолдах.

`ReviewFeatures` обучает словари и масштабирование внутри `Pipeline`, поэтому проверочные тексты не участвуют в их расчёте. Оценка поиска — **минус MAE медианы вероятностей**: обычный `predict` логистической регрессии выбрал бы самый вероятный класс, а здесь нужно другое правило.

`n_jobs=1` и два потока линейной алгебры ограничивают расход памяти. Кэш `Pipeline` повторно использует обученные преобразования при изменении только `C`. Отдельный кэш завершённого поиска зависит от данных, кода, версий библиотек и параметров.

In [3]:
from sklearn.model_selection import GridSearchCV
from threadpoolctl import threadpool_limits

from company_reviews.training import (
    build_pipeline,
    export_model,
    negative_median_mae,
    training_fingerprint,
)

configuration = {
    "tune": True, "C": 4, "min_df": 5, "threads": 2,
    "seed": 2026, "c_grid": [2, 4, 8], "min_df_grid": [2, 3, 5],
    "char_weight": 0.5, "numeric_weight": 0.1,
}
search_key, _ = training_fingerprint(root / "data/raw/train.csv", configuration)
search_dir = root / "data/cache/grid-search" / search_key
search_dir.mkdir(parents=True, exist_ok=True)
search_path = search_dir / "search.joblib"

In [4]:
memory = joblib.Memory(search_dir / "transforms", verbose=0)
search = GridSearchCV(
    estimator=build_pipeline(memory=memory),
    param_grid={
        "classifier__C": [2, 4, 8],
        "features__min_df": [2, 3, 5],
    },
    scoring=negative_median_mae,
    cv=StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2026),
    n_jobs=1,
    pre_dispatch=1,
    refit=True,
    error_score="raise",
    verbose=2,
)

In [5]:
if search_path.exists():
    search = joblib.load(search_path)
    print("Загружен завершённый поиск с теми же данными, кодом и настройками")
else:
    with threadpool_limits(limits=2):
        search.fit(
            train_clean["Review"],
            train_clean["Rating"],
            groups=normalize_reviews_for_split(train_clean["Review"]),
        )
    joblib.dump(search, search_path, compress=3)

Загружен завершённый поиск с теми же данными, кодом и настройками


In [6]:
grid_results = pd.DataFrame(search.cv_results_)
grid_table = pd.DataFrame({
    "C": grid_results["param_classifier__C"],
    "min_df": grid_results["param_features__min_df"],
    "средняя MAE": -grid_results["mean_test_score"],
    "стандартное отклонение": grid_results["std_test_score"],
})
display(grid_table.sort_values("средняя MAE").reset_index(drop=True).round(5))

best_c = search.best_params_["classifier__C"]
best_min_df = search.best_params_["features__min_df"]
print(f"Выбрано: C={best_c}; min_df={best_min_df}")
print(f"Средняя MAE: {-search.best_score_:.5f}")

,C,min_df,средняя MAE,стандартное отклонение
0,4,5,0.18539,0.00445
1,4,3,0.18557,0.00425
2,4,2,0.18561,0.00435
3,8,2,0.18579,0.00444
4,8,3,0.18614,0.00411
5,8,5,0.18676,0.00403
6,2,5,0.18686,0.00365
7,2,2,0.18709,0.00302
8,2,3,0.18734,0.00330


Выбрано: C=4; min_df=5
Средняя MAE: 0.18539


При расчёте стандартного отклонения `GridSearchCV` делит на число фолдов (5), а `pandas.std` в разделе 6 — на 4. Поэтому разброс может немного отличаться при одинаковых MAE фолдов.

Параметры и наборы признаков выбирались по этим же фолдам. Поэтому лучшая средняя MAE — результат выбора модели, **а не независимая итоговая оценка на нетронутой выборке**.

## 8. Модель для инференса

При `refit=True` поиск уже обучил лучший вариант на всех 59 976 отзывах без точных повторов. Упакуем его в `BaselineModel`: он хранит словари, масштабирование и классификатор вместе; `predict` возвращает медиану, а `predict_proba` — вероятности оценок 1–5.

In [7]:
final_model = export_model(search.best_estimator_)
model_path = root / "data/cache/baseline_model.joblib"
joblib.dump(final_model, model_path, compress=3)

loaded_model = joblib.load(model_path)
sample_reviews = train_clean["Review"].head(5)
np.testing.assert_allclose(
    loaded_model.predict_proba(sample_reviews),
    final_model.predict_proba(sample_reviews),
)

print(f"Файл модели: {model_path.relative_to(root)}")
print(f"Словных признаков: {len(final_model.word_vectorizer.vocabulary_):,}")
print(f"Символьных признаков: {len(final_model.char_vectorizer.vocabulary_):,}")
print(f"Числовых признаков: {len(final_model.numeric_columns)}")

Файл модели: data/cache/baseline_model.joblib
Словных признаков: 85,432
Символьных признаков: 71,711
Числовых признаков: 13


In [8]:
demo = pd.DataFrame({
    "отзыв": sample_reviews.str.slice(0, 100).to_numpy(),
    "реальная оценка": train_clean["Rating"].head(5).to_numpy(),
    "прогноз": loaded_model.predict(sample_reviews),
})
display(demo)

,отзыв,реальная оценка,прогноз
0,Very good value and a great tv very happy and ...,5,5
1,After 6 month still can't access my account,3,1
2,I couldn't make an official review on a produc...,1,1
3,"Fantastic! Extremely easy to use website, fant...",5,5
4,So far annoyed as hell with this bt monthly pa...,1,1


Для воспроизводимого обучения артефактов сервиса из корня проекта:

```bash
uv run --locked python -m company_reviews.training --tune
```

Команда использует тот же `GridSearchCV`, сохраняет выбранную модель и отдельный словный baseline в `models/`, записывает результаты фолдов, версии библиотек, хеш исходных данных и контрольные суммы моделей. Завершённый поиск можно повторно загрузить из локального кэша; при изменении данных, кода или настроек ключ меняется.

## Итог

Постоянная медиана, словная модель и варианты признаков сравнивались на одинаковых групповых фолдах. Числа выше показывают небольшой выигрыш инженерии признаков и настройки над сильным текстовым baseline. Эксперименты со стоп-словами, весами классов и сокращёнными наборами сохранены, чтобы был виден ход выбора. Итоговая MAE после `GridSearchCV` приведена в разделе 7; её ограничение — повторное использование фолдов для выбора модели.